# 03. Model urutan: RNN, LSTM, dan GRU

Melihat hidden state, menghubungkan RNN manual dengan nn.RNN, dan membandingkan tiga model urutan.

**Prasyarat:** modul 02.

**Pola belajar:** baca penjelasan, prediksi bentuk keluaran, jalankan kode, lalu ubah satu hal.

Contoh ulasan dalam paket ini merupakan data sintetis untuk mempelajari mekanisme. Metriknya tidak mewakili kinerja pada ulasan nyata.

## Penyiapan

Instal dependensi melalui petunjuk README sebelum menjalankan seluruh sel. Setiap notebook dapat dimulai dengan kernel baru. GPU bersifat opsional. Semua operasi tensor yang berinteraksi harus berada pada perangkat yang sesuai.

In [1]:
from pathlib import Path
import sys
# Lokal: buka dari root repo, folder nlp, atau nlp/notebooks.
# Colab: ambil paket kursus jika belum tersedia.
candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for base in candidates for p in (base, base / "nlp")
             if (p / "nlp_course").is_dir()), None)
if ROOT is None and "google.colab" in sys.modules:
    import subprocess
    target = Path("/content/pytorch-deep-learning-nlp")
    if not target.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none",
                        "--sparse", "--branch", "nlp-learning-path",
                        "https://github.com/FeliksMakarios/pytorch-deep-learning.git",
                        str(target)], check=True)
        subprocess.run(["git", "sparse-checkout", "set", "nlp"], cwd=target, check=True)
    ROOT = target / "nlp"
if ROOT is None or not (ROOT / "nlp_course").is_dir():
    raise RuntimeError("Folder nlp_course tidak ditemukan. Ikuti petunjuk README nlp.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import torch
from torch import nn
from nlp_course.data import tokenize, build_vocab, encode, read_rows, loaders, collate_batch
from nlp_course.models import MeanClassifier, RecurrentClassifier, TinyTransformer
from nlp_course.engine import seed_all, fit, run_epoch, metrics, save_mean, load_mean, predict
seed_all(42)
torch.set_num_threads(1)
device = "cuda" if torch.cuda.is_available() else "cpu"
ARTIFACTS = ROOT / "artifacts"
ARTIFACTS.mkdir(exist_ok=True)
print("PyTorch:", torch.__version__, "Perangkat:", device)

PyTorch: 2.14.0+cu130 Perangkat: cpu


## 1. Mengapa urutan diperlukan?

RNN memperbarui keadaan tersembunyi untuk setiap token. Secara sederhana, h_t = tanh(x_t W_ihᵀ + b_ih + h_(t−1) W_hhᵀ + b_hh). Keadaan terakhir merangkum urutan yang sudah dibaca. Ringkasan ini tidak menjamin semua informasi tersimpan.

In [2]:
vocab, train, val, test = loaders()
ids, lengths, labels = next(iter(train))
embedding = nn.Embedding(len(vocab),4,padding_idx=0)
x = embedding(ids[:1,:3])
print("Input bertahap:", x.shape)

Input bertahap: torch.Size([1, 3, 4])


## 2. Memeriksa satu langkah RNN manual

PyTorch menyimpan bobot dengan orientasi keluaran × masukan. Karena itu perkalian manual memakai transpose. Bandingkan hasilnya dengan keluaran nn.RNN menggunakan bobot yang sama.

In [3]:
rnn = nn.RNN(input_size=4,hidden_size=3,batch_first=True)
h = torch.zeros(1,3)
manual_states = []
for t in range(x.shape[1]):
    h = torch.tanh(x[:,t] @ rnn.weight_ih_l0.T + rnn.bias_ih_l0
                   + h @ rnn.weight_hh_l0.T + rnn.bias_hh_l0)
    manual_states.append(h)
manual = torch.stack(manual_states,dim=1)
automatic, h_n = rnn(x)
print(manual)
print(automatic)
assert torch.allclose(manual,automatic,atol=1e-6)

tensor([[[ 0.4187, -0.1558,  0.0881],
         [ 0.8564, -0.9357,  0.9793],
         [ 0.2316, -0.2883,  0.3735]]], grad_fn=<StackBackward0>)
tensor([[[ 0.4187, -0.1558,  0.0881],
         [ 0.8564, -0.9357,  0.9793],
         [ 0.2316, -0.2883,  0.3735]]], grad_fn=<TransposeBackward1>)


## 3. Padding tidak boleh menjadi token terakhir

Mengambil output pada indeks `-1` dari batch berpadded dapat mengambil posisi PAD. Kita memakai packed sequence agar RNN berhenti pada panjang asli. Tensor panjang dikirim ke CPU untuk fungsi packing.

In [4]:
from torch.nn.utils.rnn import pack_padded_sequence
packed = pack_padded_sequence(embedding(ids),lengths.cpu(),batch_first=True,enforce_sorted=False)
output, last = rnn(packed)
print("Hidden terakhir:",last.shape)
print("Panjang maksimum:",lengths.max().item())

Hidden terakhir: torch.Size([1, 16, 3])
Panjang maksimum: 6


## 4. Memahami LSTM dan GRU

LSTM mempunyai hidden state dan cell state. Gerbang mengatur informasi yang diteruskan atau diperbarui. GRU memakai struktur gerbang dengan satu keadaan utama. Keduanya membantu pembelajaran ketergantungan, tetapi hasil tetap dipengaruhi data dan optimasi.

In [5]:
lstm = nn.LSTM(4,3,batch_first=True)
_, (h_n,c_n) = lstm(packed)
gru = nn.GRU(4,3,batch_first=True)
_, g_n = gru(packed)
print("LSTM h/c:",h_n.shape,c_n.shape)
print("GRU h:",g_n.shape)

LSTM h/c: torch.Size([1, 16, 3]) torch.Size([1, 16, 3])
GRU h: torch.Size([1, 16, 3])


## 5. Membandingkan model secara terkendali

Gunakan split, kosakata, jumlah epoch, dan seed yang sama. Arsitektur berbeda memiliki jumlah parameter berbeda. Hasil satu seed merupakan demonstrasi, bukan dasar klaim bahwa satu arsitektur selalu unggul.

In [6]:
results = []
trained = {}
for kind in ["rnn","lstm","gru"]:
    seed_all(42)
    _, train, val, test = loaders(vocab=vocab,seed=42)
    candidate = RecurrentClassifier(len(vocab),dim=16,hidden=16,kind=kind)
    fit(candidate,train,val,epochs=15)
    results.append({"model":kind,"parameters":sum(p.numel() for p in candidate.parameters()),
                    **run_epoch(candidate,val)})
    trained[kind] = candidate
for row in results:
    print(row)
best_kind = min(results,key=lambda x:x["loss"])["model"]
print("Terpilih melalui validasi:",best_kind)
print("Uji:",run_epoch(trained[best_kind],test))

{'model': 'rnn', 'parameters': 1010, 'loss': 0.10177483782172203, 'accuracy': 1.0, 'macro_f1': 1.0, 'confusion': [[48, 0], [0, 48]]}
{'model': 'lstm', 'parameters': 2642, 'loss': 0.001154572208179161, 'accuracy': 1.0, 'macro_f1': 1.0, 'confusion': [[48, 0], [0, 48]]}
{'model': 'gru', 'parameters': 2098, 'loss': 0.0006779510537550474, 'accuracy': 1.0, 'macro_f1': 1.0, 'confusion': [[48, 0], [0, 48]]}
Terpilih melalui validasi: gru
Uji: {'loss': 0.0005588227601644272, 'accuracy': 1.0, 'macro_f1': 1.0, 'confusion': [[48, 0], [0, 48]]}


## 6. Uji ketahanan terhadap padding tambahan

Prediksi model packed seharusnya tidak berubah ketika PAD tambahan disisipkan di ujung batch, selama panjang asli tetap benar. Pengujian ini memeriksa mekanisme, bukan mengejar angka akurasi tertentu.

In [7]:
model = trained[best_kind].eval()
ids,lengths,_ = next(iter(val))
extra = torch.nn.functional.pad(ids,(0,5),value=0)
with torch.inference_mode():
    assert torch.allclose(model(ids,lengths),model(extra,lengths),atol=1e-6)
print("Prediksi konsisten dengan padding tambahan.")

Prediksi konsisten dengan padding tambahan.


## Latihan mandiri

1. Apa perbedaan output dan h_n pada RNN?
2. Mengapa hidden terakhir tidak boleh selalu memakai output[:,-1]?
3. Apa yang dikembalikan LSTM selain h_n?
4. Mengapa clipping gradien dipakai dalam engine?

## Pembahasan latihan

1. Output berisi keadaan pada setiap langkah, h_n berisi keadaan terakhir per lapisan dan arah.
2. Indeks itu dapat menunjuk PAD pada kalimat pendek.
3. Cell state c_n.
4. Clipping membatasi norma gradien agar pembaruan ekstrem lebih terkendali. Ini tidak menyelesaikan semua masalah optimasi.

## Penghubung ke materi berikutnya

Modul 04 berfokus pada pengolahan teks nyata dan validasi dataset, sebelum kode pelatihan dipisahkan menjadi modul Python.

### Rujukan
- [Dokumentasi PyTorch](https://docs.pytorch.org/docs/stable/index.html)
- [Sumber Embedding](https://github.com/pytorch/pytorch/blob/main/torch/nn/modules/sparse.py)
- [Sumber Transformer](https://github.com/pytorch/pytorch/blob/main/torch/nn/modules/transformer.py)
- [Kursus sumber dan struktur awal](https://github.com/mrdbourke/pytorch-deep-learning)

Materi ini ditulis sebagai jalur NLP mandiri. Penjelasan dan contoh NLP bukan terjemahan resmi kursus sumber.